# 🎙️ WAXAL ASR v2 — Multilingual Whisper, Sunbird-tuned

Upgraded with findings from *"How much speech data is necessary for ASR in African
languages?"* (Akera et al., 2025) — same model family, same Bantu language family.

**What changed from v1 and why:**

| Change | Why |
|---|---|
| **Much more data** (`PER_LANG_TRAIN` ↑↑) | Their scaling curve: 1h→47.6% WER, 50h→12.5%, 200h→9.8%. Gains are steepest early. **This is the biggest lever.** |
| **Quality filter on transcripts** | 38.6% of their high-error cases were noisy ground truth, not model failure. Cheap win most competitors skip. |
| **Audio augmentation** (noise, speed 0.9–1.1x, 5% @8kHz) | Their exact recipe. Directly targets Phase-2 unseen recordings. |
| **Beam search + repetition penalty** | Repetition loops = 4.4% of their failures; long text = 5.2%. Fixed at decode time, free. |
| **Speaker-held-out eval** | Honest generalisation signal; Phase 2 is unseen speakers. |
| **Early stopping on val loss** | Their protocol (patience-based) — stops before overfit. |

**⚠️ Undertraining is worse than not training.** Their 1-hour fine-tune (47.6% WER)
was *worse than the untouched baseline* (33.1%). Don't ship a tiny-data model.

**Runtime → Change runtime type → T4 GPU** before you start.


## 1. Setup

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ValueError:
    print("(already mounted)")

WORK_DIR = '/content/drive/MyDrive/waxal_whisper'
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)

In [ ]:
%%capture
!pip install -q -U transformers datasets accelerate jiwer evaluate librosa soundfile audiomentations
print("deps installed")

## 2. Hugging Face login
Accept the dataset terms once at https://huggingface.co/datasets/google/WaxalNLP,
then paste a **read** token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import login
login()

## 3. Config

**Read this before running.** `PER_LANG_TRAIN` is the single most important number
here. 300 was a pipeline test. 3000+ is a real run. The paper's curve says the
jump from tiny→moderate data is where nearly all the gain lives.

In [ ]:
from typing import Final

MODEL_ID: Final = "openai/whisper-small"   # -> "openai/whisper-medium" once this run is proven
DATASET_ID: Final = "google/WaxalNLP"
LANGS: Final = ["lin", "sna", "lug"]

# ---- DATA (the big lever) ----------------------------------------
PER_LANG_TRAIN: Final = 3000   # was 300. Raise toward full splits (lin~16k, sna~15k, lug~6k)
PER_LANG_EVAL: Final  = 150

# ---- QUALITY FILTER (Sunbird: 38.6% of bad cases = noisy GT) -----
MIN_CHARS: Final       = 5      # drop degenerate ultra-short refs
MAX_TOKENS_APPROX: Final = 80   # their ">80 tokens" long-text failure mode
MAX_NONLETTER_FRAC: Final = 0.15  # their >15% non-letter density rule

# ---- AUGMENTATION (their recipe) ---------------------------------
AUG_PROB: Final        = 0.5    # fraction of training clips augmented
TELEPHONE_FRAC: Final  = 0.05   # 5% downsampled to 8kHz then back

# ---- TRAINING ----------------------------------------------------
BATCH: Final       = 8      # 4 for medium
GRAD_ACCUM: Final  = 4      # effective batch 32 (their setting)
MAX_STEPS: Final   = 2000   # was 400
SAVE_EVERY: Final  = 250
EVAL_EVERY: Final  = 250
LR: Final          = 1e-5   # their LR; 5e-6 for large
EARLY_STOP_PATIENCE: Final = 4   # evals without improvement

# ---- DECODING (fixes their repetition/long-text failures) --------
NUM_BEAMS: Final        = 5
REPETITION_PENALTY: Final = 1.15
NO_REPEAT_NGRAM: Final  = 4

OUT_DIR = f"{WORK_DIR}/whisper-waxal-v2"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("model:", MODEL_ID, "| per-lang train:", PER_LANG_TRAIN, "| out:", OUT_DIR)

## 4. Normalization + quality filter

The filter implements the paper's three noisy-ground-truth heuristics: explicit
noise markers, high non-letter density, and degenerate short references.

In [ ]:
import re

NOISE_MARKER = re.compile(r"[\[\(<]\s*(noise|inaudible|unclear|silence|music|laugh|cough|unintelligible)[^\]\)>]*[\]\)>]", re.I)

def normalize_text(t: str) -> str:
    t = str(t).strip()
    return re.sub(r"\s+", " ", t)

def is_good_transcript(t: str) -> bool:
    """Sunbird noisy-ground-truth heuristics."""
    if t is None: return False
    t = str(t).strip()
    if len(t) < MIN_CHARS: return False                      # degenerate short
    if NOISE_MARKER.search(t): return False                  # explicit noise marker
    if len(t.split()) > MAX_TOKENS_APPROX: return False       # very long text
    letters = sum(ch.isalpha() or ch.isspace() for ch in t)
    if letters / max(len(t), 1) < (1 - MAX_NONLETTER_FRAC):  # non-letter density
        return False
    return True

# quick sanity check
for s in ["Mombe idzo dziri kuratidza", "[noise]", "ok", "###@@@!!! ###@@@"]:
    print(repr(s), "->", is_good_transcript(s))

## 5. Stream all three languages, filtered

We over-fetch (`* 1.3`) so that after dropping bad transcripts we still land near
the target count. `speaker_id` is kept for held-out validation.

In [ ]:
from datasets import load_dataset, get_dataset_config_names, Audio, Dataset, concatenate_datasets

configs = get_dataset_config_names(DATASET_ID)
print("configs:", [c for c in configs if any(c.startswith(l) for l in LANGS)])

def cfg_for(lang):
    cands = sorted([c for c in configs if c.startswith(lang)],
                   key=lambda c: ("tts" in c, "asr" not in c, len(c)))
    return cands[0] if cands else f"{lang}_asr"

def stream_take(lang, split, n):
    cfg = cfg_for(lang)
    s = load_dataset(DATASET_ID, cfg, split=split, streaming=True)
    s = s.cast_column("audio", Audio(sampling_rate=16_000))
    rows, seen, dropped = [], 0, 0
    for ex in s:
        seen += 1
        if len(rows) >= n: break
        if seen > n * 3: break            # safety valve
        if not is_good_transcript(ex.get("transcription")):
            dropped += 1; continue
        rows.append({
            "audio": ex["audio"]["array"],
            "transcription": ex["transcription"],
            "language": lang,
            "speaker_id": str(ex.get("speaker_id", f"{lang}_spk_{len(rows)}")),
        })
        if len(rows) % 250 == 0: print(f"  {lang}/{split}: kept {len(rows)}/{n} (dropped {dropped})")
    print(f"  {lang}/{split}: FINAL kept {len(rows)}, dropped {dropped} bad transcripts")
    return Dataset.from_list(rows)

train_parts, eval_parts = [], []
for lang in LANGS:
    train_parts.append(stream_take(lang, "train", PER_LANG_TRAIN))
    try:
        eval_parts.append(stream_take(lang, "validation", PER_LANG_EVAL))
    except Exception:
        eval_parts.append(stream_take(lang, "train", PER_LANG_EVAL))

train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
eval_ds  = concatenate_datasets(eval_parts)
print("TOTAL train:", len(train_ds), "| eval:", len(eval_ds))

### 5b. Speaker-held-out check

If training and eval share speakers, your local score flatters you and Phase 2
will disappoint. This removes any eval speaker that appears in training.

In [ ]:
train_speakers = set(train_ds["speaker_id"])
before = len(eval_ds)
eval_ds = eval_ds.filter(lambda ex: ex["speaker_id"] not in train_speakers)
print(f"eval: {before} -> {len(eval_ds)} after removing speakers seen in training")
if len(eval_ds) < 30:
    print("⚠️  Few unseen speakers left — speaker_id may be missing/unique per clip.")
    print("    Local score will be optimistic; weight Phase-2 robustness accordingly.")

## 6. Audio augmentation — the Sunbird recipe

Random noise injection, speed perturbation 0.9–1.1x, and 5% downsampled to 8 kHz
to simulate telephone-quality speech. Applied **only to training**, never eval.
This is your main insurance against Phase 2's unseen recording conditions.

In [ ]:
import numpy as np, librosa

rng = np.random.default_rng(42)

def augment(arr: np.ndarray) -> np.ndarray:
    a = np.asarray(arr, dtype=np.float32)

    # 1. speed perturbation 0.9-1.1x
    if rng.random() < 0.5:
        rate = float(rng.uniform(0.9, 1.1))
        a = librosa.effects.time_stretch(a, rate=rate)

    # 2. random noise injection
    if rng.random() < 0.5:
        snr_db = float(rng.uniform(10, 30))
        rms = np.sqrt(np.mean(a ** 2)) + 1e-9
        noise_rms = rms / (10 ** (snr_db / 20))
        a = a + rng.normal(0, noise_rms, a.shape).astype(np.float32)

    # 3. telephone simulation: down to 8 kHz and back
    if rng.random() < TELEPHONE_FRAC / AUG_PROB:
        a = librosa.resample(a, orig_sr=16_000, target_sr=8_000)
        a = librosa.resample(a, orig_sr=8_000, target_sr=16_000)

    return np.clip(a, -1.0, 1.0)

# visual sanity check on one clip
_s = np.asarray(train_ds[0]["audio"], dtype=np.float32)
print("original len:", len(_s), "-> augmented len:", len(augment(_s)))

## 7. Processor + feature prep (augment train only)

In [ ]:
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_ID)

def make_prepare(do_augment: bool):
    def prepare(batch):
        arr = np.asarray(batch["audio"], dtype=np.float32)
        if do_augment and rng.random() < AUG_PROB:
            arr = augment(arr)
        batch["input_features"] = processor.feature_extractor(
            arr, sampling_rate=16_000).input_features[0]
        batch["labels"] = processor.tokenizer(
            normalize_text(batch["transcription"])).input_ids
        return batch
    return prepare

train_prep = train_ds.map(make_prepare(True),  remove_columns=train_ds.column_names)
eval_prep  = eval_ds.map(make_prepare(False), remove_columns=eval_ds.column_names)
print("prepared — train:", len(train_prep), "| eval:", len(eval_prep))

## 8. Data collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

## 9. Load Whisper — language-agnostic, SpecAugment, anti-repetition decoding

`language = None` is what makes this Phase-2 viable: the model detects language
from the signal instead of being told. The beam/repetition settings target the
paper's two decode-time failure modes.

In [ ]:
from transformers import WhisperForConditionalGeneration
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

# Language-agnostic (Phase 2 gives no language metadata)
model.generation_config.language = None
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# SpecAugment (on top of the audio-level augmentation in section 6)
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

# Anti-repetition / long-text decoding
model.generation_config.num_beams = NUM_BEAMS
model.generation_config.repetition_penalty = REPETITION_PENALTY
model.generation_config.no_repeat_ngram_size = NO_REPEAT_NGRAM

collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)
print("model ready:", MODEL_ID, "| beams:", NUM_BEAMS)

## 10. Metric = 0.5·WER + 0.5·CER

In [ ]:
import evaluate
wer_m = evaluate.load("wer"); cer_m = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = [normalize_text(p) for p in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_text(l) for l in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    keep = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not keep: return {"wer": 1.0, "cer": 1.0, "score": 1.0}
    pred_str, label_str = zip(*keep)
    wer = wer_m.compute(predictions=list(pred_str), references=list(label_str))
    cer = cer_m.compute(predictions=list(pred_str), references=list(label_str))
    return {"wer": wer, "cer": cer, "score": 0.5*wer + 0.5*cer}

## 11. Train — early stopping, checkpoints to Drive

Effective batch = `BATCH × GRAD_ACCUM` = 32, matching the paper. Early stopping
(patience 4 evals) mirrors their protocol and prevents overfitting the small
languages. Resumes automatically after a disconnect.

In [ ]:
from transformers import (Seq2SeqTrainingArguments, Seq2SeqTrainer,
                          EarlyStoppingCallback)
import glob

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=50,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=SAVE_EVERY,
    eval_steps=EVAL_EVERY,
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="score",
    greater_is_better=False,
    save_total_limit=2,
    seed=42,
)

trainer = Seq2SeqTrainer(
    args=args, model=model,
    train_dataset=train_prep, eval_dataset=eval_prep,
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"),
               key=lambda p: int(p.split("-")[-1]))
resume = ckpts[-1] if ckpts else None
print("resuming from:", resume)
trainer.train(resume_from_checkpoint=resume)

trainer.save_model(f"{OUT_DIR}/final")
processor.save_pretrained(f"{OUT_DIR}/final")
print("✅ saved ->", f"{OUT_DIR}/final")

### 11b. Baseline comparison — is fine-tuning actually helping?

The paper's sharpest warning: their 1-hour fine-tune (47.6% WER) was **worse than
the untouched model** (33.1%). Run this to confirm your fine-tune beats the
off-the-shelf baseline. If it doesn't, you need more data, not more steps.

In [ ]:
import torch, numpy as np

def quick_score(mdl, proc, ds, n=40):
    mdl.eval()
    preds, refs = [], []
    for ex in ds.select(range(min(n, len(ds)))):
        feats = proc.feature_extractor(np.asarray(ex["audio"], dtype=np.float32),
                                       sampling_rate=16_000,
                                       return_tensors="pt").input_features.to(mdl.device)
        with torch.no_grad():
            ids = mdl.generate(feats, max_new_tokens=225, num_beams=NUM_BEAMS,
                               repetition_penalty=REPETITION_PENALTY,
                               no_repeat_ngram_size=NO_REPEAT_NGRAM)
        preds.append(normalize_text(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]))
        refs.append(normalize_text(ex["transcription"]))
    w = wer_m.compute(predictions=preds, references=refs)
    c = cer_m.compute(predictions=preds, references=refs)
    return w, c, 0.5*w + 0.5*c

base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(model.device)
base.generation_config.language = None
base.generation_config.forced_decoder_ids = None
bw, bc, bs = quick_score(base, processor, eval_ds)
del base; torch.cuda.empty_cache()

fw, fc, fs = quick_score(model, processor, eval_ds)

print(f"BASELINE  (no fine-tune): WER={bw:.3f} CER={bc:.3f} score={bs:.3f}")
print(f"FINE-TUNED             : WER={fw:.3f} CER={fc:.3f} score={fs:.3f}")
print("✅ fine-tuning helped" if fs < bs else
      "⚠️  fine-tune is WORSE than baseline — raise PER_LANG_TRAIN before submitting")

---
# 🏁 Inference → submission.csv

Language-agnostic and phase-portable. Beam search + repetition penalty applied.

In [ ]:
import pandas as pd
TEST_CSV = "Test.csv"; SAMPLE_CSV = "SampleSubmission.csv"
test = pd.read_csv(TEST_CSV)
id_col = test.columns[0]
print("test ids:", len(test), "| example:", test[id_col].iloc[0])

In [ ]:
from datasets import load_dataset, Audio

def build_lookup():
    lookup, need = {}, set(test[id_col])
    for lang in LANGS:
        if not need: break
        cfg = cfg_for(lang)
        for split in ["test", "validation", "train"]:
            if not need: break
            try:
                s = load_dataset(DATASET_ID, cfg, split=split, streaming=True)
                s = s.cast_column("audio", Audio(sampling_rate=16_000))
                for ex in s:
                    if ex["id"] in need:
                        lookup[ex["id"]] = ex["audio"]["array"]
                        need.discard(ex["id"])
                        if len(lookup) % 250 == 0: print(f"  found {len(lookup)}/{len(test)}")
                        if not need: break
            except Exception as e:
                print(f"  {lang}/{split}: {str(e)[:80]}")
    print("resolved", len(lookup), "/", len(test))
    return lookup

audio_lookup = build_lookup()

In [ ]:
import torch, numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor

m = WhisperForConditionalGeneration.from_pretrained(f"{OUT_DIR}/final").to(
    "cuda" if torch.cuda.is_available() else "cpu").eval()
proc = WhisperProcessor.from_pretrained(f"{OUT_DIR}/final")

def transcribe(arr):
    feats = proc.feature_extractor(np.asarray(arr, dtype=np.float32),
                                   sampling_rate=16_000,
                                   return_tensors="pt").input_features.to(m.device)
    with torch.no_grad():
        ids = m.generate(feats, max_new_tokens=225,
                         num_beams=NUM_BEAMS,
                         repetition_penalty=REPETITION_PENALTY,
                         no_repeat_ngram_size=NO_REPEAT_NGRAM)
    return normalize_text(proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0])

rows = []
for i, _id in enumerate(test[id_col]):
    arr = audio_lookup.get(_id)
    rows.append((_id, transcribe(arr) if arr is not None else ""))
    if (i+1) % 200 == 0: print(f"  transcribed {i+1}/{len(test)}")
print("transcribed", len(rows))

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
sid, stgt = sample.columns[0], sample.columns[1]
sample[stgt] = sample[sid].map(dict(rows)).fillna("")
empty = (sample[stgt].str.strip() == "").sum()
sample.to_csv(f"{WORK_DIR}/submission.csv", index=False)
sample.to_csv("submission.csv", index=False)
print(f"wrote submission.csv: {len(sample)} rows, {empty} empty")
sample.head()

## Roadmap — in strict priority order

1. **Run this as-is** (3000/lang, whisper-small). Check section 11b: fine-tune must beat baseline.
2. **Raise `PER_LANG_TRAIN`** toward the full splits (lin ~16k, sna ~15k, lug ~6k). Paper says
   this is where the gain is — steepest returns before ~200h equivalent.
3. **Upgrade to `whisper-medium`** (`BATCH=4`). Then large-v3 only if you get better GPU
   than a free T4 — 1.55B params won't fine-tune conventionally on 16GB.
4. **Add public data** (allowed, must disclose): Common Voice / FLEURS for lin/sna/lug.
   Broader speakers = better Phase-2 generalisation.
5. **Trust your local speaker-held-out score**, not the Phase-1 leaderboard.
   Phase 2 decides prizes.

**Honest ceiling note:** the paper's numbers come from H100/A100 training on
large-v3 with hundreds of hours. On a free T4 with whisper-small/medium you will
not match 7% WER. Aim to maximise what this setup can reach, and prioritise
generalisation over leaderboard chasing.
